In [1]:
from pathlib import Path
import xmltodict
import pandas as pd
import numpy as np
import tifffile as tiff
import xmltodict
from dateutil import parser
import re

import matplotlib.pyplot as plt
from collections import namedtuple
from shapely.geometry import box

from skimage.registration import phase_cross_correlation
from skimage.transform import rescale, resize, downscale_local_mean
from skimage import exposure

import zarr
from pylibCZIrw import czi as pyczi

In [2]:
"""Skip during test
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA23\Intermediate")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA23\Proximal")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA23\Distal")
section_folders = []
for folder in series_folder.iterdir():  # Iterate over all items in the folder
    if folder.is_dir() and folder.name.startswith("S_"):
        print(f"fould series folder: {folder.name}")
        section_folders.append(folder)
"""

'Skip during test\nseries_folder = Path(r"E:\\PROJECTS\\EM\\LUKE\\TA23\\Intermediate")\nseries_folder = Path(r"E:\\PROJECTS\\EM\\LUKE\\TA23\\Proximal")\nseries_folder = Path(r"E:\\PROJECTS\\EM\\LUKE\\TA23\\Distal")\nsection_folders = []\nfor folder in series_folder.iterdir():  # Iterate over all items in the folder\n    if folder.is_dir() and folder.name.startswith("S_"):\n        print(f"fould series folder: {folder.name}")\n        section_folders.append(folder)\n'

In [3]:
from atlas.io import is_there_a_single_tif, extract_s_number

In [4]:
series_folder = Path(r"Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10-1ROI-w1-20nm-bsd")

#TODO: important change this to metadata readout
pixel_size = {
    'Value': 0.020,
    'Axial': 0.20,
    'Unit': 'µm'
}
""" EXAMPLE
with open(tif_folder.joinpath('tif-stack-metadata.xml'), "rb") as f:
    metadata_dict = xmltodict.parse(f, xml_attribs=True)

pixel_size = metadata_dict['human_meta_data']['pixel_size']
"""

tif_list = []
for folder in series_folder.iterdir():  # Iterate over all items in the folder
    if folder.is_file() and folder.name.endswith(".tiff"):
        print(f"fould series image: {folder.name}")
        tif_list.append(folder)

fould series image: stitched_image_S_1.tiff
fould series image: stitched_image_S_10.tiff
fould series image: stitched_image_S_11.tiff
fould series image: stitched_image_S_12.tiff
fould series image: stitched_image_S_13.tiff
fould series image: stitched_image_S_14.tiff
fould series image: stitched_image_S_15.tiff
fould series image: stitched_image_S_16.tiff
fould series image: stitched_image_S_17.tiff
fould series image: stitched_image_S_18.tiff
fould series image: stitched_image_S_19.tiff
fould series image: stitched_image_S_2.tiff
fould series image: stitched_image_S_20.tiff
fould series image: stitched_image_S_21.tiff
fould series image: stitched_image_S_22.tiff
fould series image: stitched_image_S_23.tiff
fould series image: stitched_image_S_24.tiff
fould series image: stitched_image_S_25.tiff
fould series image: stitched_image_S_26.tiff
fould series image: stitched_image_S_27.tiff
fould series image: stitched_image_S_28.tiff
fould series image: stitched_image_S_29.tiff
fould series

In [5]:
"""SKIP during test
tif_list = []
for section_folder in section_folders:
    single_tif, tif_path = is_there_a_single_tif(section_folder)
    if single_tif:
        #print(tif_path)
        tif_list.append(tif_path)
"""

tif_list_sorted = sorted(tif_list, key=extract_s_number)

In [6]:
from atlas.io.fibics_metadata import extract_tif_metadata, get_pixel_size_from_tif

In [7]:
"""SKIP during test
#TODO: important change this to metadata readout
pixel_size = {
    'Value': get_pixel_size_from_tif(tif_list_sorted[0].name, tif_list_sorted[0].parent),
    'Axial': 0.07,
    'Unit': 'µm'
}
"""

""" EXAMPLE
with open(tif_folder.joinpath('tif-stack-metadata.xml'), "rb") as f:
    metadata_dict = xmltodict.parse(f, xml_attribs=True)

pixel_size = metadata_dict['human_meta_data']['pixel_size']
"""
print("work on pixel size")

work on pixel size


In [8]:
from atlas.io import create_empty_folder, rm_tree
from atlas.image_analysis import image_dtype_min_max, mask_low_and_saturation, rescale_image_intensity
from atlas.alignment import calculate_cumulative_shifts, initialize_alignment_df, pairwise_alignment

In [9]:
output_path = series_folder.joinpath("alignment_results")
create_empty_folder(output_path)

down_scale = 10
tif_list_sorted = sorted(tif_list, key=extract_s_number)

# initialize the dataframe, in particular we asign the pair-wise patching of the images
# based on the sorted list of tiff.
z_align_df = initialize_alignment_df(tif_list_sorted, down_scale)
# run pairwise alignment based on the provided dataframe
z_align_df = pairwise_alignment(z_align_df)
# now we can calculate the cumulative shifts
z_align_df = calculate_cumulative_shifts(z_align_df)

# ✅ At the end, save the DataFrame as a CSV for later analysis
z_align_df.to_csv(output_path.joinpath("z_alignment_results.csv"), index=False)

z_align_df 

Created new (or emptied) folder: Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10-1ROI-w1-20nm-bsd\alignment_results
Processing alignment: ref -> stitched_image_S_1.tiff, moving -> stitched_image_S_1.tiff
crop pixel shift: [0 0]
Detected pixel offset based on crops (row, col): [0. 0.]
current shift: [0. 0.]
Processing alignment: ref -> stitched_image_S_1.tiff, moving -> stitched_image_S_2.tiff
crop pixel shift: [-39 150]
Detected pixel offset based on crops (row, col): [2. 6.]
current shift: [-37. 156.]
Processing alignment: ref -> stitched_image_S_2.tiff, moving -> stitched_image_S_3.tiff
crop pixel shift: [0 2]
Detected pixel offset based on crops (row, col): [-10.  -2.]
current shift: [-10.   0.]
Processing alignment: ref -> stitched_image_S_3.tiff, moving -> stitched_image_S_4.tiff
crop pixel shift: [ 0 -2]
Detected pixel offset based on crops (row, col): [ 2. -4.]
current shift: [ 2. -6.]
Processing alignment: ref -> stitched_image_S_4.tiff, moving -> stitched_image_S_5.tiff
crop pi

KeyboardInterrupt: 

In [ ]:
# drop sections that are out of place, I need a better procedure for this, in this case easy because they were at the end
# Drop the last 2 rows in-place
#z_align_df.drop(z_align_df.index[-2:], inplace=True)
# TODO: work on a code to drop based on index values, however this might require re-run of alignment.
# a solution would be to uncouple the pairwise alignment from the comulative one, so this would not require
# a full rerun but just a couple of extra operations

In [ ]:
from atlas.io import apply_alignment

use_down_sample = True
zarr_array, zarr_path = apply_alignment(z_align_df, buffer_pixels=20, use_down_sample=use_down_sample)

In [ ]:
from atlas.io import zarr_array_to_czi

In [ ]:
out_pixel_size = pixel_size.copy()

if use_down_sample:
    out_pixel_size['Value'] = pixel_size['Value'] * down_scale
    end_str = "_ds_aligned"
else:
    end_str = "_aligned"

czi_path = zarr_array_to_czi(zarr_path, out_pixel_size, end_str=end_str)

In [ ]:
if zarr_path.exists():
  rm_tree(zarr_path)

In [ ]:
use_down_sample = False
zarr_array, zarr_path = apply_alignment(z_align_df, buffer_pixels=20, use_down_sample=use_down_sample)

In [ ]:
out_pixel_size = pixel_size.copy()

if use_down_sample:
    out_pixel_size['Value'] = pixel_size['Value'] * down_scale
    end_str = "_ds_aligned"
else:
    end_str = "_aligned"

czi_path = zarr_array_to_czi(zarr_path, out_pixel_size, end_str=end_str)